# Байесовские нейросети и оценка неопределённости

**Проект «ИИ для учёных».** Практический блокнот к странице алгоритма «Байесовские нейросети и оценка неопределённости».

## Что делает метод

Обычная нейронная сеть на каждый входной объект возвращает одно точечное значение — предсказание. При работе вне области обучающих данных (новые диапазоны параметров, нетипичные образцы) такая сеть всё равно выдаёт уверенный ответ, хотя её предсказание может быть полностью ненадёжным.

Байесовские нейросети и близкие практические методы — MC Dropout и глубокие ансамбли — устраняют этот изъян: вместо одного числа модель возвращает распределение прогнозов. Ширина этого распределения и есть мера неопределённости.

В этом блокноте мы:

1. Обучим нейросеть с MC Dropout и небольшой глубокий ансамбль на задаче 1D-регрессии.
2. Покажем, как неопределённость честно возрастает вне области обучения.
3. Разложим полную неопределённость на эпистемическую (от нехватки данных) и алеаторную (от шума в самих данных).
4. Построим диаграмму калибровки (reliability diagram) и вычислим ECE.

Расчётное время прохождения на CPU — около 8–12 минут.

## Интуиция за методом

Представьте, что вы отправляете запрос независимым экспертам, а не одному. Если все дают близкие оценки — результату можно доверять. Если мнения расходятся — задача трудная или данных недостаточно, и само расхождение информативно.

Именно так работают практические методы байесовской нейросетевой регрессии:

- **MC Dropout** превращает одну сеть в ансамбль случайных подсетей. Dropout, активный не только при обучении, но и при инференсе, каждый раз случайно «выключает» часть нейронов. T прогонов дают T разных ответов — их разброс и есть мера неопределённости.

- **Глубокий ансамбль** буквально обучает несколько (3–5) независимых сетей с разными случайными инициализациями. Расхождение их предсказаний отражает, насколько сильно исход зависит от выбора начального состояния — то есть от нехватки данных для однозначной подгонки.

**Два типа неопределённости, которые важно различать:**

- **Эпистемическая** (от греч. «знание») — неопределённость от нехватки данных. Снижается, если добавить наблюдения. Именно её показывает расхождение ансамблевых голосов. Важна для активного обучения: туда и стоит направить следующий дорогой эксперимент.

- **Алеаторная** (от лат. «случай») — неопределённость, заложенная в самих данных: шум измерительного прибора, квантовые флуктуации, биологическая изменчивость. Её не устранить, получив больше наблюдений того же типа.

**Калибровка** — ключевая проверка качества оценок неопределённости. Хорошо откалиброванная модель: если она говорит «с вероятностью 80 % истина в этом интервале», то реально 80 % наблюдений попадают в него. Плохо откалиброванная сеть (обычно переобученная) говорит «90 % уверенность» там, где реальная точность 60 % — это опасно для научных выводов.

## 1. Установка библиотек

В среде Google Colab перечисленные библиотеки, как правило, уже установлены. Ячейка ниже гарантирует их наличие, в том числе при локальном запуске.

In [ ]:
%pip install -q torch==2.3.1 numpy==1.26.4 matplotlib==3.9.2 scikit-learn==1.5.1

## 2. Единый стиль графиков

Все графики в блокнотах проекта оформляются в едином визуальном стиле сайта «ИИ для учёных»: общая палитра, шрифты, толщины линий и сетка. Ниже встроено содержимое стилевого шаблона `notebooks/viz_style.py` — отдельный файл загружать не нужно. Вызов `apply_viz_style()` настраивает matplotlib один раз на весь блокнот.

In [ ]:
# Единый стилевой шаблон графиков проекта «ИИ для учёных».
# Значения синхронизированы с токенами темы сайта (styles/tokens.css).
VIZ_TOKENS = {
    "background": "#ffffff",  # фон полотна
    "node_fill":  "#eef1f6",  # заливка блоков
    "node_text":  "#0e1726",  # основной текст
    "edge":       "#4d5e78",  # линии, оси, подписи
    "grid":       "#dce2ec",  # сетка координат
    "series":     ["#2563eb", "#0d9488", "#b45309", "#4d5e78"],  # цвета рядов
}
VIZ = VIZ_TOKENS


def apply_viz_style():
    """Настраивает matplotlib под единый стиль сайта."""
    import matplotlib as mpl
    from cycler import cycler
    t = VIZ_TOKENS
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Segoe UI", "DejaVu Sans", "Arial", "Helvetica"],
        "font.size": 12, "axes.titlesize": 15, "axes.titleweight": "semibold",
        "axes.labelsize": 13, "xtick.labelsize": 11, "ytick.labelsize": 11,
        "legend.fontsize": 11,
        "figure.facecolor": t["background"], "axes.facecolor": t["background"],
        "savefig.facecolor": t["background"], "figure.dpi": 110, "savefig.dpi": 150,
        "axes.prop_cycle": cycler(color=t["series"]),
        "text.color": t["node_text"], "axes.labelcolor": t["node_text"],
        "axes.titlecolor": t["node_text"], "axes.edgecolor": t["edge"],
        "xtick.color": t["edge"], "ytick.color": t["edge"], "axes.linewidth": 1.2,
        "axes.grid": True, "grid.color": t["grid"], "grid.linewidth": 1.0,
        "grid.linestyle": "-", "axes.axisbelow": True,
        "lines.linewidth": 2.0, "lines.markersize": 6, "patch.linewidth": 1.5,
        "axes.spines.top": False, "axes.spines.right": False,
        "legend.frameon": False,
    })


def get_palette(n=None):
    """Возвращает список цветов рядов из токенов темы."""
    series = VIZ_TOKENS["series"]
    if n is None:
        return list(series)
    return [series[i % len(series)] for i in range(n)]


apply_viz_style()
print("Единый стиль графиков подключён.")

## 3. Демонстрационные данные

Сгенерируем зашумлённые наблюдения нелинейной функции с намеренным **пробелом** в центре диапазона — область `[-0.5, 1.5]` не содержит обучающих точек. Это позволяет наглядно проверить главное свойство метода: хорошо откалиброванная нейросеть с оценкой неопределённости должна давать широкий доверительный коридор именно в пробеле, а узкий — там, где данных много.

Уровень шума задан параметром `sigma_noise`. Он имитирует алеаторную неопределённость — шум прибора или биологическую изменчивость, которую нельзя устранить сбором большего числа точек.

In [ ]:
import numpy as np
import torch

# Фиксируем все генераторы случайности для воспроизводимости.
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# Истинная функция, скрытая от модели.
def true_fn(x):
    return np.sin(2.5 * x) + 0.4 * x

sigma_noise = 0.25  # стандартное отклонение шума измерений (алеаторная неопределённость)

# Обучающие точки: левый кластер и правый, пробел в [-0.5, 1.5].
rng = np.random.default_rng(SEED)
x_left  = rng.uniform(-3.0, -0.5, 18)
x_right = rng.uniform( 1.5,  3.0, 18)
x_train = np.sort(np.concatenate([x_left, x_right])).astype(np.float32)
y_train = (true_fn(x_train)
           + rng.normal(0, sigma_noise, x_train.size)).astype(np.float32)

# Плотная сетка для визуализации — охватывает всю область, включая пробел.
x_grid = np.linspace(-4.0, 4.0, 400).astype(np.float32)

print(f"Обучающих точек: {x_train.size}")
print(f"Пробел в данных: [-0.5, 1.5] — {x_train.size} точек обучения,"
      f" 0 в пробеле.")

## 4. Применение метода

### Шаг 4.1. Архитектура сети и вспомогательные функции

Строим небольшую полносвязную сеть: вход 1 → два скрытых слоя по 64 нейрона → выход 2 нейрона.

Почему два выходных нейрона? Нейросеть предсказывает не только среднее значение, но и **логарифм дисперсии** — это стандартный приём для оценки алеаторной неопределённости. Модель обучается максимизировать правдоподобие нормального распределения с предсказанным средним и дисперсией (гетероскедастическая регрессия).

**Dropout** включён в режиме инференса (`model.train()` при вызове `predict`) — это и есть MC Dropout. Каждый прогон использует разную маску выключенных нейронов, имитируя случайную подсеть.

In [ ]:
import torch
import torch.nn as nn


class HeteroscedasticNet(nn.Module):
    """
    Небольшая полносвязная сеть для гетероскедастической регрессии.

    Предсказывает среднее (mu) и логарифм дисперсии (log_var) одновременно.
    Dropout активен при инференсе — это ключевой шаг MC Dropout.
    """

    def __init__(self, hidden: int = 64, dropout_p: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, hidden),
            nn.ReLU(),
            nn.Dropout(p=dropout_p),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Dropout(p=dropout_p),
        )
        self.head_mu      = nn.Linear(hidden, 1)  # предсказание среднего
        self.head_log_var = nn.Linear(hidden, 1)  # предсказание log(sigma^2)

    def forward(self, x: torch.Tensor):
        h = self.net(x)
        mu      = self.head_mu(h).squeeze(-1)
        log_var = self.head_log_var(h).squeeze(-1)
        return mu, log_var


def nll_loss(mu: torch.Tensor,
             log_var: torch.Tensor,
             y: torch.Tensor) -> torch.Tensor:
    """
    Отрицательное лог-правдоподобие нормального распределения.

    Потеря = 0.5 * (log_var + (y - mu)^2 / exp(log_var))
    Обучение этой функции заставляет сеть предсказывать честную дисперсию:
    занижение дисперсии штрафуется вторым слагаемым, завышение — первым.
    """
    var = torch.exp(log_var).clamp(min=1e-6)
    return (0.5 * (log_var + (y - mu).pow(2) / var)).mean()


print("Архитектура определена.")
print(HeteroscedasticNet())

### Шаг 4.2. Обучение одной сети (MC Dropout)

Обучим одну сеть, которую затем будем использовать в режиме MC Dropout.

In [ ]:
def train_net(seed: int,
              hidden: int = 64,
              dropout_p: float = 0.1,
              lr: float = 3e-3,
              n_epochs: int = 800) -> HeteroscedasticNet:
    """
    Обучает одну сеть и возвращает обученную модель.

    seed  — начальное состояние для воспроизводимости.
    Малое число нейронов и короткий цикл обучения обеспечивают
    работу на CPU за разумное время (~15–30 с на сеть).
    """
    torch.manual_seed(seed)
    net = HeteroscedasticNet(hidden=hidden, dropout_p=dropout_p)
    opt = torch.optim.Adam(net.parameters(), lr=lr)

    Xt = torch.from_numpy(x_train).unsqueeze(1)
    yt = torch.from_numpy(y_train)

    for epoch in range(1, n_epochs + 1):
        net.train()
        opt.zero_grad()
        mu, lv = net(Xt)
        loss = nll_loss(mu, lv, yt)
        loss.backward()
        opt.step()

        if epoch % 200 == 0:
            print(f"  эпоха {epoch:4d}: потеря NLL = {loss.item():.4f}")
    return net


print("Обучение MC-Dropout-сети (seed=42)...")
mc_net = train_net(seed=42)
print("Готово.")

### Шаг 4.3. MC Dropout: T прогонов с активным Dropout

Ключевая идея MC Dropout — при инференсе сеть остаётся в режиме `train()`, поэтому Dropout активен и каждый прогон использует другую случайную маску нейронов. T прогонов порождают T разных предсказаний.

Из этих T значений:
- **Среднее mu** → итоговый прогноз.
- **Дисперсия средних** → **эпистемическая неопределённость** (от нехватки данных).
- **Среднее предсказанных дисперсий** → **алеаторная неопределённость** (шум в данных).
- **Полная неопределённость** = сумма обеих составляющих.

In [ ]:
def mc_predict(net: HeteroscedasticNet,
               x: np.ndarray,
               T: int = 100,
               seed: int = 0):
    """
    MC Dropout инференс: T прогонов с активным Dropout.

    Возвращает:
        mu_mean   — итоговое предсказание (среднее по T прогонам)
        std_epist — стандартное отклонение, отражающее эпистемическую неопределённость
        std_aleat — стандартное отклонение, отражающее алеаторную неопределённость
        std_total — полное стандартное отклонение
    """
    torch.manual_seed(seed)
    Xt = torch.from_numpy(x).unsqueeze(1)

    mus, vars_ = [], []
    # net.train() оставляет Dropout активным — это принципиально важно!
    net.train()
    with torch.no_grad():
        for _ in range(T):
            mu, log_var = net(Xt)
            mus.append(mu.numpy())
            vars_.append(np.exp(log_var.numpy()))

    mus   = np.stack(mus)    # форма (T, N)
    vars_ = np.stack(vars_)  # форма (T, N)

    mu_mean   = mus.mean(axis=0)
    var_epist = mus.var(axis=0)           # дисперсия средних прогнозов по T прогонам
    var_aleat = vars_.mean(axis=0)        # среднее предсказанных дисперсий

    std_epist = np.sqrt(var_epist)
    std_aleat = np.sqrt(var_aleat)
    std_total = np.sqrt(var_epist + var_aleat)

    return mu_mean, std_epist, std_aleat, std_total


T = 100  # число MC-прогонов; увеличьте до 200–500 для более гладких оценок
mu_mc, std_epist_mc, std_aleat_mc, std_total_mc = mc_predict(
    mc_net, x_grid, T=T)

print(f"MC Dropout ({T} прогонов) завершён.")
print(f"Средняя эпистемическая std: {std_epist_mc.mean():.4f}")
print(f"Средняя алеаторная    std: {std_aleat_mc.mean():.4f}")

### Шаг 4.4. Глубокий ансамбль: 5 независимых сетей

Обучим 5 сетей с разными случайными инициализациями. Разброс их предсказаний — мера эпистемической неопределённости. Это самый эмпирически надёжный метод по данным сравнительных исследований (Lakshminarayanan et al., 2017; Ovadia et al., 2019).

In [ ]:
M = 5  # число членов ансамбля; при M=3 быстрее, при M=10 — надёжнее

print(f"Обучение глубокого ансамбля ({M} сетей)...")
ensemble = []
for m in range(M):
    print(f"  Сеть {m + 1}/{M}:")
    net_m = train_net(seed=100 + m)
    ensemble.append(net_m)
    print(f"  Сеть {m + 1} обучена.")

print("Ансамбль готов.")

In [ ]:
def ensemble_predict(nets, x: np.ndarray):
    """
    Инференс глубокого ансамбля.

    Каждая сеть запускается в режиме eval() (Dropout выключен) —
    здесь ансамбль, а не MC Dropout, обеспечивает разнообразие.
    Разложение неопределённости аналогично MC Dropout.
    """
    Xt = torch.from_numpy(x).unsqueeze(1)
    mus, vars_ = [], []
    for net in nets:
        net.eval()
        with torch.no_grad():
            mu, log_var = net(Xt)
            mus.append(mu.numpy())
            vars_.append(np.exp(log_var.numpy()))

    mus   = np.stack(mus)    # форма (M, N)
    vars_ = np.stack(vars_)  # форма (M, N)

    mu_mean   = mus.mean(axis=0)
    var_epist = mus.var(axis=0)    # расхождение предсказаний членов ансамбля
    var_aleat = vars_.mean(axis=0) # средняя алеаторная дисперсия

    std_epist = np.sqrt(var_epist)
    std_aleat = np.sqrt(var_aleat)
    std_total = np.sqrt(var_epist + var_aleat)

    return mu_mean, std_epist, std_aleat, std_total


mu_ens, std_epist_ens, std_aleat_ens, std_total_ens = ensemble_predict(
    ensemble, x_grid)

print(f"Ансамбль: средняя эпистемическая std = {std_epist_ens.mean():.4f}")
print(f"Ансамбль: средняя алеаторная    std = {std_aleat_ens.mean():.4f}")

## 5. Интерпретация результата

### График 5.1. Прогнозы и полоса неопределённости: MC Dropout vs. ансамбль

Ключевой результат блокнота — сравнение двух методов на одном наборе данных. Для каждого отображаем:
- линию предсказания (среднее по прогонам или членам ансамбля),
- узкую полосу ±1 std (около 68 % покрытие),
- широкую полосу ±2 std (около 95 % покрытие).

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, 2, figsize=(14.0, 5.4), sharey=True)

configs = [
    ("MC Dropout (T=100)",    x_grid, mu_mc,  std_total_mc),
    (f"Глубокий ансамбль (M={M})", x_grid, mu_ens, std_total_ens),
]

for ax, (title, xg, mu, std) in zip(axes, configs):
    # Истинная функция
    ax.plot(xg, true_fn(xg), color=VIZ['series'][2], linestyle='--',
            linewidth=1.8, label='истинная функция', zorder=3)
    # Предсказание
    ax.plot(xg, mu, color=VIZ['series'][0], linewidth=2.2,
            label='прогноз', zorder=4)
    # Полосы неопределённости
    ax.fill_between(xg, mu - 2 * std, mu + 2 * std,
                    color=VIZ['series'][0], alpha=0.13,
                    label='±2 std (~95 %)')
    ax.fill_between(xg, mu - std, mu + std,
                    color=VIZ['series'][0], alpha=0.25,
                    label='±1 std (~68 %)')
    # Обучающие точки
    ax.scatter(x_train, y_train, color=VIZ['series'][3], s=28, zorder=5,
               label='обучающие точки')
    # Пометка области пробела
    ax.axvspan(-0.5, 1.5, color=VIZ['grid'], alpha=0.55, zorder=0)
    ax.text(0.5, ax.get_ylim()[0] if hasattr(ax, '_ybound') else -2.8,
            'пробел', ha='center', va='bottom',
            color=VIZ['edge'], fontsize=10)

    ax.set_title(title)
    ax.set_xlabel('x')

axes[0].set_ylabel('y')
axes[0].legend(loc='upper left', fontsize=10)

fig.suptitle('Прогноз с оценкой неопределённости: вне обучающей области — широкий интервал',
             fontsize=13, y=1.02)
fig.tight_layout()
plt.show()

**Как читать эти графики.**

Серая полоса в центре — пробел в обучающих данных (`[-0.5, 1.5]`). Синяя линия — предсказание, синие полосы — неопределённость (±1 std и ±2 std). Пунктир — истинная функция.

Что нужно проверить:

1. В областях левее -0.5 и правее 1.5 (где данных много) полоса должна быть узкой.
2. В пробеле [-0.5, 1.5] полоса заметно расширяется — это правильное поведение.
3. За пределами [-4, -3] и [3, 4] (экстраполяция) полоса расширяется ещё сильнее.
4. Истинная функция должна большей частью укладываться в полосу ±2 std.

Если видите нарушение п. 2 (пробел не даёт широкой полосы) — попробуйте увеличить T или M.

### График 5.2. Разложение неопределённости: эпистемическая vs. алеаторная

Покажем, как полная неопределённость разбивается на две составляющие для ансамблевой модели. Это разложение критически важно для активного обучения: нас интересует именно эпистемическая составляющая — именно она снизится при добавлении новых измерений.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11.0, 7.0), sharex=True)

# Верхняя панель: предсказание + полоса полной неопределённости.
ax = axes[0]
ax.plot(x_grid, true_fn(x_grid), color=VIZ['series'][2],
        linestyle='--', linewidth=1.8, label='истинная функция')
ax.plot(x_grid, mu_ens, color=VIZ['series'][0],
        linewidth=2.2, label='прогноз ансамбля')
ax.fill_between(x_grid,
                mu_ens - 2 * std_total_ens,
                mu_ens + 2 * std_total_ens,
                color=VIZ['series'][0], alpha=0.18,
                label='±2 std (полная)')
ax.scatter(x_train, y_train, color=VIZ['series'][3], s=28, zorder=5,
           label='обучающие точки')
ax.axvspan(-0.5, 1.5, color=VIZ['grid'], alpha=0.5, zorder=0)
ax.set_ylabel('y')
ax.set_title('Ансамбль: предсказание с полной неопределённостью')
ax.legend(fontsize=10, loc='upper left')

# Нижняя панель: эпистемическая и алеаторная составляющие.
ax = axes[1]
ax.plot(x_grid, std_epist_ens, color=VIZ['series'][0],
        linewidth=2.0, label='эпистемическая std')
ax.plot(x_grid, std_aleat_ens, color=VIZ['series'][1],
        linewidth=2.0, label='алеаторная std')
ax.plot(x_grid, std_total_ens, color=VIZ['series'][2],
        linewidth=1.6, linestyle='--', label='полная std')
ax.axvspan(-0.5, 1.5, color=VIZ['grid'], alpha=0.5, zorder=0)
ax.set_xlabel('x')
ax.set_ylabel('стандартное отклонение')
ax.set_title('Разложение неопределённости: эпистемическая растёт в пробеле, алеаторная постоянна')
ax.legend(fontsize=10)

fig.tight_layout()
plt.show()

**Как читать этот график.**

Нижняя панель показывает стандартные отклонения двух типов неопределённости:

- **Синяя линия (эпистемическая)**: должна быть высокой в пробеле [-0.5, 1.5] и низкой там, где данных много. Это мера нехватки информации — именно туда стоит направить следующий эксперимент при активном обучении.

- **Зелёная линия (алеаторная)**: должна быть примерно постоянной по всему диапазону. Она отражает шум в самих данных (у нас `sigma_noise = 0.25`) — её нельзя снизить, добавив новые наблюдения.

- **Пунктирная линия (полная)**: сумма обеих составляющих.

### График 5.3. Диаграмма калибровки (reliability diagram) и ECE

Калибровка проверяет, насколько предсказанные интервалы неопределённости соответствуют реальному покрытию. Для этого генерируем независимую проверочную выборку и смотрим: если модель говорит «90 % вероятность», попадает ли реально 90 % точек в предсказанный интервал?

**ECE (Expected Calibration Error)** — взвешенное среднее абсолютных отклонений реального покрытия от заявленного. ECE = 0 — идеальная калибровка; чем меньше, тем лучше.

In [ ]:
from scipy import stats as sp_stats

# Генерируем проверочную выборку: x по всему диапазону, y = истинная функция + шум.
rng_val = np.random.default_rng(99)
x_val = rng_val.uniform(-3.0, 3.0, 300).astype(np.float32)
y_val = (true_fn(x_val)
         + rng_val.normal(0, sigma_noise, x_val.size)).astype(np.float32)

# Предсказание ансамбля на проверочной выборке.
mu_val, _, _, std_val = ensemble_predict(ensemble, x_val)


def reliability_diagram(mu, std, y_true, n_bins=10, label=''):
    """
    Вычисляет реальное покрытие для набора уровней доверия
    и возвращает (уровни, покрытия, ECE).
    """
    levels = np.linspace(0.05, 0.95, n_bins)
    coverages = []
    for level in levels:
        # z-квантиль для нормального распределения
        z = sp_stats.norm.ppf((1 + level) / 2)
        lo = mu - z * std
        hi = mu + z * std
        coverage = np.mean((y_true >= lo) & (y_true <= hi))
        coverages.append(coverage)
    coverages = np.array(coverages)
    ece = np.mean(np.abs(coverages - levels))
    return levels, coverages, ece


# Калибровка MC Dropout на проверочной выборке.
mu_val_mc, _, _, std_val_mc = mc_predict(mc_net, x_val, T=100)
lev_mc,  cov_mc,  ece_mc  = reliability_diagram(mu_val_mc,  std_val_mc,  y_val, label='MC Dropout')
lev_ens, cov_ens, ece_ens = reliability_diagram(mu_val,     std_val,     y_val, label='Ансамбль')

fig, ax = plt.subplots(figsize=(7.0, 6.5))
ax.plot([0, 1], [0, 1], color=VIZ['edge'], linestyle='--',
        linewidth=1.8, label='идеальная калибровка')
ax.plot(lev_mc,  cov_mc,  'o-', color=VIZ['series'][0],
        linewidth=2.0, markersize=7, label=f'MC Dropout  (ECE={ece_mc:.3f})')
ax.plot(lev_ens, cov_ens, 's-', color=VIZ['series'][1],
        linewidth=2.0, markersize=7, label=f'Ансамбль    (ECE={ece_ens:.3f})')
ax.set_xlabel('Заявленный уровень доверия')
ax.set_ylabel('Реальное покрытие')
ax.set_title('Диаграмма калибровки (reliability diagram)')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.legend()
fig.tight_layout()
plt.show()

print(f"ECE MC Dropout: {ece_mc:.4f}")
print(f"ECE Ансамбль:   {ece_ens:.4f}")
print("(меньше — лучше; 0 — идеальная калибровка)")

**Как читать диаграмму калибровки.**

Ось X — уровень доверия, заявленный моделью (например, 0.8 означает «80 % предиктивный интервал»). Ось Y — реальная доля точек, попавших в этот интервал. Пунктирная диагональ — идеальная калибровка.

- **Кривая выше диагонали**: модель **недоуверена** (underconfident) — заявляет меньше, чем есть на самом деле, даёт избыточно широкие интервалы.
- **Кривая ниже диагонали**: модель **переуверена** (overconfident) — заявляет больше, чем есть, её интервалы слишком узки.

ECE — число, которое позволяет сравнивать методы количественно: чем ближе к нулю, тем лучше откалибрована модель.

**Важно**: публикуя оценки неопределённости в научной работе, всегда включайте reliability diagram и ECE на независимой выборке — без этого оценки неопределённости нельзя считать обоснованными.

## 6. Попробуйте на своих данных

Замените синтетические данные своими наблюдениями.

**Требования к данным:**
- Одна или несколько входных переменных (признаков), одна целевая.
- Минимум ~30–50 наблюдений. При очень малой выборке неопределённость будет высокой везде.
- Не обязательно 1D: для многомерного X замените `nn.Linear(1, hidden)` на `nn.Linear(n_features, hidden)`.

**Шаги:**

1. Загрузите файл в Colab через панель «Файлы» (иконка папки слева).
2. Снимите комментарии в ячейке ниже.
3. Повторно выполните разделы 4 и 5.

In [ ]:
# --- Шаблон загрузки своих данных ---
# import pandas as pd
# from sklearn.preprocessing import StandardScaler
#
# df = pd.read_csv('my_data.csv')
# feature_col = 'имя_признака'     # или список признаков
# target_col  = 'имя_цели'
#
# x_raw = df[[feature_col]].to_numpy(dtype='float32')
# y_raw = df[target_col].to_numpy(dtype='float32')
#
# # Нормировка обязательна для стабильного обучения нейросети.
# scaler_x = StandardScaler().fit(x_raw)
# scaler_y = StandardScaler().fit(y_raw.reshape(-1, 1))
# x_train = scaler_x.transform(x_raw).squeeze().astype('float32')
# y_train = scaler_y.transform(y_raw.reshape(-1, 1)).squeeze().astype('float32')
#
# # Для многомерного входа замените nn.Linear(1, hidden) на nn.Linear(X.shape[1], hidden)
# # в классе HeteroscedasticNet.
#
# # Далее повторите разделы 4 и 5, подставив свои x_train, y_train, x_grid.

## 7. Поэкспериментируйте сами

| Параметр | Что изменить | Что наблюдать |
|---|---|---|
| `T` в `mc_predict` | 10 vs. 200 | Как меняется гладкость оценки неопределённости? |
| `M` (число членов ансамбля) | 1 vs. 10 | При M=1 нет разброса — исчезает ли эпистемическая составляющая? |
| `dropout_p` в `HeteroscedasticNet` | 0.05 vs. 0.5 | Как меняется ширина полосы MC Dropout в пробеле? |
| `sigma_noise` | 0.05 vs. 1.0 | Меняется ли алеаторная составляющая? Растёт ли ECE? |
| Размер пробела | `[-1.0, 2.0]` (шире) | Становится ли пробел виден сильнее на графике? |
| `n_epochs` | 200 vs. 2000 | Улучшается ли калибровка при более долгом обучении? |

**Дополнительный эксперимент.** Удалите пробел в данных (используйте равномерную выборку по всему диапазону). Убедитесь, что эпистемическая неопределённость снизилась во всём диапазоне, а алеаторная осталась прежней — это подтверждает корректность разложения.

## Краткие выводы

- Обычная нейронная сеть выдаёт уверенный точечный прогноз даже вне области обучения — это опасно для научных приложений.
- **MC Dropout**: Dropout, оставленный активным при инференсе, и T прогонов дают распределение предсказаний. Просто в реализации, работает поверх любой существующей сети.
- **Глубокий ансамбль**: M независимо обученных сетей — эмпирически лучший метод по качеству калибровки, но требует в M раз больше вычислений.
- **Эпистемическая неопределённость** растёт вне области обучения и снижается при добавлении данных — именно её нужно использовать для активного обучения.
- **Алеаторная неопределённость** отражает шум в данных — её нельзя снизить никаким увеличением выборки.
- **Калибровка** (reliability diagram, ECE) обязательна: без неё заявленные интервалы неопределённости не имеют количественного смысла.
- Обе составляющие оцениваются через гетероскедастическую потерю (NLL нормального распределения) — сеть учится предсказывать и среднее, и дисперсию одновременно.

## Три вопроса для самопроверки

Ответьте до того, как раскроете подсказку, — это проверяет, что метод понят, а не просто запущен.

1. В вашем эксперименте MC Dropout при T=10 даёт заметно неровную кривую эпистемической неопределённости, а при T=100 — гладкую. Почему, и как выбрать достаточное T для реальной задачи?

2. Вы применили ансамбль из M=5 сетей к новому набору данных и получили ECE = 0.18 (реальное покрытие систематически ниже заявленного). Назовите два практических способа улучшить калибровку, не меняя архитектуру и не добавляя данных.

3. Коллега утверждает, что для задач активного обучения нужно выбирать следующую точку эксперимента там, где максимальна **полная** неопределённость (эпистемическая + алеаторная). Почему это неверно, и что правильнее использовать в качестве критерия выбора?

<details>
<summary>Показать ориентиры для ответов</summary>

1. При малом T оценка дисперсии прогнозов строится по небольшому числу образцов — она нестабильна (высокая дисперсия оценки дисперсии). При T=100 выборочная оценка стабилизируется. Практическое правило: увеличивайте T до тех пор, пока кривая неопределённости перестаёт заметно меняться при повторном запуске (изменяйте seed и сравнивайте кривые). Обычно T=50–100 достаточно для сглаживания; T=200–500 — для публикационного качества.

2. Два способа улучшить калибровку без изменения архитектуры: (а) **температурное масштабирование** (temperature scaling) — подобрать скалярный параметр T>1, делящий log_var (или добавляемый к нему), минимизируя NLL на отложенной выборке; это стандартный пост-обучающий шаг, не меняющий точность; (б) **увеличить число членов ансамбля** M, поскольку большее разнообразие ансамбля расширяет предиктивные интервалы и снижает overconfidence. Также можно проверить наличие утечки информации из проверочной выборки в обучение.

3. Алеаторная неопределённость не снижается от добавления наблюдений — это присущий данным шум. Максимизировать полную неопределённость при выборе следующей точки означает часто выбирать области с высоким шумом, а не области нехватки информации. Правильный критерий — **эпистемическая составляющая** (дисперсия средних по ансамблю / по MC-прогонам): именно она снижается при получении нового измерения. В байесовской оптимизации используют взаимную информацию (mutual information) между предсказанием и весами — это строгий аналог эпистемической неопределённости.
</details>